### Seq2Seq com Attention — MIMO (WRSO)

Treina um Seq2Seq (encoder-decoder LSTM) com dot-attention multivariado, prevendo `y1..y3` (vazão de distribuição, nível, pressão) a partir de uma janela de `u1..u4` + `y1..y3`. Mesmos dados usados no ARMAX/MOGWO (`notebooks/modeling/01_mogwo_armax_reproducao.ipynb`), gerados por `notebooks/processing/01_mimo_dataset_from_papper.ipynb`. Arquitetura baseada em `data/external/seq2seq attention.ipynb`.

In [1]:
from pathlib import Path

import lightning as L
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torchmetrics
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
L.seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_PROCESSED = Path("../../data/processed")
MODELS_DIR = Path("../../models")
FIGURES_DIR = Path("../../reports/figures/training")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

INPUT_COLS = ["freq_b1", "freq_b2", "freq_b3", "vazao_entrada"]
OUTPUT_COLS = ["vazao_distribuicao", "nivel_res", "pressao"]
FEATURE_COLS = INPUT_COLS + OUTPUT_COLS

INPUT_WINDOW = 24
OUTPUT_WINDOW = 12

/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/28 21:35:09 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Seed set to 42


### Dados

Mesmos splits do artigo usados no ARMAX: `u1..u4` = frequências das bombas + vazão de entrada; `y1..y3` = vazão de distribuição, nível e pressão. A janela de entrada usa entradas e saídas medidas; a janela de saída prevê só `y1..y3`.

In [2]:
train_df = pd.read_parquet(
    DATA_PROCESSED / "mimo_dataset_sample_from_papper_train.parquet"
)
test_df = pd.read_parquet(
    DATA_PROCESSED / "mimo_dataset_sample_from_papper_test.parquet"
)

train_raw = train_df[FEATURE_COLS].to_numpy()
test_raw = test_df[FEATURE_COLS].to_numpy()

print("treino:", train_raw.shape, "teste:", test_raw.shape)

treino: (601, 7) teste: (501, 7)


In [3]:
scaler = MinMaxScaler().fit(train_raw)
train_scaled = scaler.transform(train_raw)
test_scaled = scaler.transform(test_raw)

N_TARGETS = len(OUTPUT_COLS)
target_idx = [FEATURE_COLS.index(c) for c in OUTPUT_COLS]

In [4]:
class WindowDataset(Dataset):
    """Sliding-window (input, target) pairs over a multivariate series.

    `data` holds every feature (inputs + outputs); the target window
    keeps only `target_idx` columns.
    """

    def __init__(
        self,
        data: np.ndarray,
        target_idx: list[int],
        input_window: int,
        output_window: int,
    ) -> None:
        self.data = data
        self.target_idx = target_idx
        self.input_window = input_window
        self.output_window = output_window
        self.n_samples = (
            len(data) - input_window - output_window + 1
        )

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(
        self, idx: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        start = idx
        mid = start + self.input_window
        end = mid + self.output_window
        x = self.data[start:mid]
        y = self.data[mid:end][:, self.target_idx]
        return torch.FloatTensor(x), torch.FloatTensor(y)


train_dataset = WindowDataset(
    train_scaled, target_idx, INPUT_WINDOW, OUTPUT_WINDOW
)
val_size = max(1, int(0.15 * len(train_dataset)))
train_size = len(train_dataset) - val_size
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
test_dataset = WindowDataset(
    test_scaled, target_idx, INPUT_WINDOW, OUTPUT_WINDOW
)

train_loader = DataLoader(
    train_subset, batch_size=32, shuffle=True, pin_memory=True
)
val_loader = DataLoader(
    val_subset, batch_size=32, shuffle=False, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False, pin_memory=True
)

x_batch, y_batch = next(iter(train_loader))
print("x_batch:", x_batch.shape, "y_batch:", y_batch.shape)

x_batch: torch.Size([32, 24, 7]) y_batch: torch.Size([32, 12, 3])


### Modelo Seq2Seq com dot-attention

Encoder LSTM sobre a janela de entrada (todas as features); decoder LSTM autoregressivo com dot-attention sobre os estados do encoder, prevendo as `N_TARGETS` saídas passo a passo. Porte multivariado de `data/external/seq2seq attention.ipynb`.

In [5]:
class Encoder(nn.Module):
    def __init__(
        self, input_size: int, hidden_size: int, num_layers: int = 1
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=False,
        )

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        outputs, (hidden, cell) = self.lstm(x)
        return outputs, hidden, cell


class DotAttention(nn.Module):
    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.fc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(
        self, hidden: torch.Tensor, encoder_outputs: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        hidden_last_layer = hidden[-1]  # [batch, hidden]
        attn_weights = torch.bmm(
            encoder_outputs.permute(1, 0, 2),
            hidden_last_layer.unsqueeze(2),
        ).squeeze(2)
        attn_weights = F.softmax(attn_weights, dim=1)

        context = torch.bmm(
            attn_weights.unsqueeze(1),
            encoder_outputs.permute(1, 0, 2),
        ).squeeze(1)

        output = torch.cat((context, hidden_last_layer), dim=1)
        output = torch.tanh(self.fc(output))
        return output, attn_weights


class Decoder(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int = 1,
    ) -> None:
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=False,
        )
        self.attention = DotAttention(hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(
        self,
        x: torch.Tensor,
        hidden: torch.Tensor,
        cell: torch.Tensor,
        encoder_outputs: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        _, (hidden, cell) = self.lstm(x, (hidden, cell))
        attn_output, attn_weights = self.attention(hidden, encoder_outputs)
        prediction = self.fc(attn_output)
        return prediction, hidden, cell, attn_weights


class Seq2SeqAttention(nn.Module):
    """Encoder-decoder LSTM with dot-attention for MIMO forecasting.

    The decoder feeds back its own prediction (padded with zeros on
    the non-target columns, since only `output_size` of `input_size`
    features are predicted) unless teacher forcing kicks in.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        output_window: int,
        num_layers: int = 1,
    ) -> None:
        super().__init__()
        self.encoder = Encoder(input_size, hidden_size, num_layers)
        self.decoder = Decoder(
            input_size, hidden_size, output_size, num_layers
        )
        self.output_window = output_window
        self.input_size = input_size
        self.output_size = output_size

    def forward(
        self,
        source: torch.Tensor,
        target: torch.Tensor | None = None,
        teacher_forcing_ratio: float = 0.5,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        seq_len, batch_size, _ = source.shape

        outputs = torch.zeros(
            self.output_window,
            batch_size,
            self.output_size,
            device=source.device,
        )
        attention_weights = torch.zeros(
            self.output_window, batch_size, seq_len, device=source.device
        )

        encoder_outputs, hidden, cell = self.encoder(source)
        decoder_input = source[-1].unsqueeze(0)  # [1, batch, input_size]

        for t in range(self.output_window):
            decoder_output, hidden, cell, attn_weights = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs[t] = decoder_output
            attention_weights[t] = attn_weights

            use_teacher_forcing = (
                target is not None
                and torch.rand(1).item() < teacher_forcing_ratio
            )
            if use_teacher_forcing:
                next_input = source[-1].clone()
                next_input[:, : self.output_size] = target[t]
                decoder_input = next_input.unsqueeze(0)
            else:
                next_input = source[-1].clone()
                next_input[:, : self.output_size] = decoder_output
                decoder_input = next_input.unsqueeze(0)

        return outputs, attention_weights

### LightningModule

Wrapper de treino/validação/teste com MSE loss, `torchmetrics.MeanSquaredError`, `MeanAbsoluteError` e `R2Score` (por saída) em cada etapa, `AdamW` + `ReduceLROnPlateau`.

In [6]:
class Seq2SeqLightning(L.LightningModule):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        output_window: int,
        num_layers: int = 1,
        lr: float = 1e-3,
        teacher_forcing_ratio: float = 0.5,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()
        self.model = Seq2SeqAttention(
            input_size=input_size,
            hidden_size=hidden_size,
            output_size=output_size,
            output_window=output_window,
            num_layers=num_layers,
        )
        self.criterion = nn.MSELoss()
        self.teacher_forcing_ratio = teacher_forcing_ratio

        self.train_mse = torchmetrics.MeanSquaredError()
        self.val_mse = torchmetrics.MeanSquaredError()
        self.test_mse = torchmetrics.MeanSquaredError()
        self.train_mae = torchmetrics.MeanAbsoluteError()
        self.val_mae = torchmetrics.MeanAbsoluteError()
        self.test_mae = torchmetrics.MeanAbsoluteError()
        self.train_r2 = torchmetrics.R2Score(multioutput="raw_values")
        self.val_r2 = torchmetrics.R2Score(multioutput="raw_values")
        self.test_r2 = torchmetrics.R2Score(multioutput="raw_values")

    def _shared_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], teacher_forcing: float
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x, y = batch
        x = x.permute(1, 0, 2)  # [seq_len, batch, features]
        y = y.permute(1, 0, 2)  # [output_window, batch, targets]
        outputs, _ = self.model(
            x, y, teacher_forcing_ratio=teacher_forcing
        )
        loss = self.criterion(outputs, y)
        return loss, outputs.contiguous(), y.contiguous()

    def _log_r2(
        self, stage: str, r2_metric: torchmetrics.R2Score,
        outputs: torch.Tensor, y: torch.Tensor,
    ) -> None:
        # R2Score expects [N, num_outputs]; flatten output_window+batch.
        flat_outputs = outputs.reshape(-1, outputs.shape[-1])
        flat_y = y.reshape(-1, y.shape[-1])
        r2_metric(flat_outputs, flat_y)
        for name, value in zip(OUTPUT_COLS, r2_metric.compute()):
            self.log(f"{stage}_r2_{name}", value, on_epoch=True, on_step=False)

    def training_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(
            batch, self.teacher_forcing_ratio
        )
        self.train_mse(outputs, y)
        self.train_mae(outputs, y)
        self._log_r2("train", self.train_r2, outputs, y)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_mse", self.train_mse, on_epoch=True, on_step=False)
        self.log("train_mae", self.train_mae, on_epoch=True, on_step=False)
        return loss

    def validation_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, teacher_forcing=0.0)
        self.val_mse(outputs, y)
        self.val_mae(outputs, y)
        self._log_r2("val", self.val_r2, outputs, y)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_mse", self.val_mse, on_epoch=True, on_step=False)
        self.log("val_mae", self.val_mae, on_epoch=True, on_step=False)
        return loss

    def test_step(
        self, batch: tuple[torch.Tensor, torch.Tensor], batch_idx: int
    ) -> torch.Tensor:
        loss, outputs, y = self._shared_step(batch, teacher_forcing=0.0)
        self.test_mse(outputs, y)
        self.test_mae(outputs, y)
        self._log_r2("test", self.test_r2, outputs, y)
        self.log("test_loss", loss)
        self.log("test_mse", self.test_mse, on_epoch=True, on_step=False)
        self.log("test_mae", self.test_mae, on_epoch=True, on_step=False)
        return loss

    def configure_optimizers(self) -> dict:
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
            },
        }

### Treino

In [7]:
HIDDEN_SIZE = 64
NUM_LAYERS = 2

lit_model = Seq2SeqLightning(
    input_size=len(FEATURE_COLS),
    hidden_size=HIDDEN_SIZE,
    output_size=N_TARGETS,
    output_window=OUTPUT_WINDOW,
    num_layers=NUM_LAYERS,
    lr=1e-3,
    teacher_forcing_ratio=0.5,
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=20),
    ModelCheckpoint(monitor="val_mae", mode="min", save_top_k=1),
]

trainer = L.Trainer(
    max_epochs=200,
    accelerator="auto",
    callbacks=callbacks,
    logger=False,
    enable_progress_bar=True,
)

mlflow.pytorch.autolog()
mlflow.set_experiment("mimo_sys_seq2seq_attention")

with mlflow.start_run():
    mlflow.log_params(
        {
            "input_window": INPUT_WINDOW,
            "output_window": OUTPUT_WINDOW,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "seed": SEED,
        }
    )
    trainer.fit(lit_model, train_loader, val_loader)
    test_metrics = trainer.test(
        lit_model, test_loader, ckpt_path="best"
    )

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True


INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


2026/09/28 21:35:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/rwp/code/mimo_sys/.venv/lib/python3.12/site-packages/mlflow/pytorch/_lightning_autolog.py:542: UserWarning: Autologging is known to be compatible with pytorch-lightning versions between 2.4.0 and 2.6.5 and may not succeed with packages outside this range."


INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


...

Testing DataLoader 0:  87%|████████▋ | 13/15 [00:00<00:00, 96.14it/s]

Testing DataLoader 0:  93%|█████████▎| 14/15 [00:00<00:00, 96.91it/s]

Testing DataLoader 0: 100%|██████████| 15/15 [00:00<00:00, 92.40it/s]

Testing DataLoader 0: 100%|██████████| 15/15 [00:00<00:00, 91.78it/s]


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric               DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            0.008874078281223774
         test_mae            0.06766891479492188
         test_mse            0.008874078281223774
    test_r2_nivel_res          0.26105996966362
     test_r2_pressao          0.6109107136726379
test_r2_vazao_distribuicao    0.8195395469665527
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


### Avaliação na escala original

Desfaz o `MinMaxScaler` (colunas de `OUTPUT_COLS`) para comparar predição x medido nas unidades do artigo.

In [8]:
best_model = Seq2SeqLightning.load_from_checkpoint(
    callbacks[1].best_model_path
)
best_model.eval()
best_model.to(device)

predictions, actuals, attn_weights_all = [], [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.permute(1, 0, 2).to(device)
        y_batch_perm = y_batch.permute(1, 0, 2).to(device)
        outputs, attn_weights = best_model.model(
            x_batch, teacher_forcing_ratio=0.0
        )
        predictions.append(outputs.permute(1, 0, 2).cpu())
        actuals.append(y_batch_perm.permute(1, 0, 2).cpu())
        attn_weights_all.append(attn_weights.permute(1, 0, 2).cpu())

predictions = torch.cat(predictions, dim=0).numpy()
actuals = torch.cat(actuals, dim=0).numpy()
attn_weights_all = torch.cat(attn_weights_all, dim=0).numpy()
print("predictions:", predictions.shape, "actuals:", actuals.shape)
print("attention:", attn_weights_all.shape)

predictions: (466, 12, 3) actuals: (466, 12, 3)
attention: (466, 12, 24)


In [9]:
target_min = scaler.data_min_[target_idx]
target_range = scaler.data_range_[target_idx]


def inverse_targets(arr: np.ndarray) -> np.ndarray:
    """Undo MinMaxScaler on the target columns only."""
    return arr * target_range + target_min


predictions_orig = inverse_targets(predictions)
actuals_orig = inverse_targets(actuals)

flat_predictions = torch.from_numpy(
    predictions_orig.reshape(-1, N_TARGETS)
)
flat_actuals = torch.from_numpy(actuals_orig.reshape(-1, N_TARGETS))

mse_metric = torchmetrics.MeanSquaredError(num_outputs=N_TARGETS)
mae_metric = torchmetrics.MeanAbsoluteError()
r2_metric = torchmetrics.R2Score(multioutput="raw_values")

mse_per_output = mse_metric(flat_predictions, flat_actuals)
mae_per_output = torch.stack(
    [
        torchmetrics.functional.mean_absolute_error(
            flat_predictions[:, i], flat_actuals[:, i]
        )
        for i in range(N_TARGETS)
    ]
)
r2_per_output = r2_metric(flat_predictions, flat_actuals)

for name, mse, mae, r2 in zip(
    OUTPUT_COLS, mse_per_output, mae_per_output, r2_per_output
):
    print(f"{name}: MSE={mse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

vazao_distribuicao: MSE=85.9302  MAE=6.9784  R²=0.8684
nivel_res: MSE=0.0337  MAE=0.1386  R²=0.3707
pressao: MSE=3.8819  MAE=1.4861  R²=0.7241


In [10]:
labels = ["Vazão de distribuição (l/s)", "Nível (m)", "Pressão (mca)"]
sample_idx = 0

fig = make_subplots(rows=3, cols=1, subplot_titles=labels)
for i, label in enumerate(labels):
    fig.add_trace(
        go.Scatter(
            y=actuals_orig[sample_idx, :, i],
            name="Medido",
            line=dict(color="black"),
            legendgroup="medido",
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            y=predictions_orig[sample_idx, :, i],
            name="Predito",
            line=dict(color="firebrick"),
            legendgroup="predito",
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
fig.update_layout(
    title_text="Seq2Seq com Attention — uma janela de teste",
    height=700,
    template="plotly_white",
)
fig.show()

### Forecast completo do conjunto de teste

Reconstrói a previsão do teste inteiro encadeando janelas não sobrepostas de `OUTPUT_WINDOW` passos (mesma lógica de `plot_full_test_prediction` da referência), para comparar com o medido ao longo de toda a série.

In [11]:
def forecast_full_series(
    model: Seq2SeqLightning,
    data_scaled: np.ndarray,
    input_window: int,
    output_window: int,
) -> np.ndarray:
    """Chain non-overlapping output_window forecasts over a series."""
    model.eval()
    data_tensor = torch.FloatTensor(data_scaled).to(device)
    preds = []

    n_steps = (len(data_scaled) - input_window) // output_window
    for step in range(n_steps):
        start = step * output_window
        window = data_tensor[start : start + input_window]
        window = window.unsqueeze(1)  # [seq_len, batch=1, features]
        with torch.no_grad():
            output, _ = model.model(window, teacher_forcing_ratio=0.0)
        preds.append(output.squeeze(1).cpu().numpy())

    return np.concatenate(preds, axis=0)


full_predictions_scaled = forecast_full_series(
    best_model, test_scaled, INPUT_WINDOW, OUTPUT_WINDOW
)
full_predictions = inverse_targets(full_predictions_scaled)
full_actuals = test_raw[
    INPUT_WINDOW : INPUT_WINDOW + len(full_predictions_scaled)
][:, target_idx]

full_predictions_t = torch.from_numpy(full_predictions)
full_actuals_t = torch.from_numpy(full_actuals)

full_mse = torchmetrics.MeanSquaredError(num_outputs=N_TARGETS)(
    full_predictions_t, full_actuals_t
)
full_mae = torch.stack(
    [
        torchmetrics.functional.mean_absolute_error(
            full_predictions_t[:, i], full_actuals_t[:, i]
        )
        for i in range(N_TARGETS)
    ]
)
full_r2 = torchmetrics.R2Score(multioutput="raw_values")(
    full_predictions_t, full_actuals_t
)
for name, mse, mae, r2 in zip(OUTPUT_COLS, full_mse, full_mae, full_r2):
    print(f"{name}: MSE={mse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

fig = make_subplots(rows=3, cols=1, subplot_titles=labels)
for i, label in enumerate(labels):
    fig.add_trace(
        go.Scatter(
            y=full_actuals[:, i],
            name="Medido",
            line=dict(color="black"),
            legendgroup="medido",
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            y=full_predictions[:, i],
            name="Predito",
            line=dict(color="royalblue"),
            legendgroup="predito",
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
    fig.update_yaxes(title_text=label, row=i + 1, col=1)
fig.update_layout(
    title_text="Seq2Seq com Attention — forecast do teste completo",
    height=800,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

vazao_distribuicao: MSE=87.6402  MAE=7.0145  R²=0.8659
nivel_res: MSE=0.0387  MAE=0.1487  R²=0.2799
pressao: MSE=4.0123  MAE=1.5218  R²=0.7137


### Pesos de attention

Heatmap e gráficos de barra dos pesos de attention de uma janela de teste — quanto cada passo de entrada pesou na previsão de cada passo de saída (mesma visualização de `plot_attention` da referência).

In [12]:
attn_sample = attn_weights_all[sample_idx]  # [output_window, input_window]

fig = go.Figure(
    data=go.Heatmap(
        z=attn_sample,
        colorscale="Viridis",
        colorbar=dict(title="Peso"),
    )
)
fig.update_layout(
    title_text="Pesos de attention — janela de teste",
    xaxis_title="Passo de entrada",
    yaxis_title="Passo de saída",
    template="plotly_white",
    height=500,
)
fig.show()

In [13]:
n_steps_to_plot = min(OUTPUT_WINDOW, 5)
fig = make_subplots(
    rows=n_steps_to_plot,
    cols=1,
    subplot_titles=[
        f"Passo de saída {t + 1}" for t in range(n_steps_to_plot)
    ],
)
for t in range(n_steps_to_plot):
    fig.add_trace(
        go.Bar(
            x=np.arange(INPUT_WINDOW),
            y=attn_sample[t],
            marker_color="steelblue",
            showlegend=False,
        ),
        row=t + 1,
        col=1,
    )
fig.update_layout(
    title_text="Pesos de attention por passo de saída",
    height=180 * n_steps_to_plot,
    template="plotly_white",
)
fig.update_xaxes(title_text="Passo de entrada", row=n_steps_to_plot, col=1)
fig.show()

In [14]:
torch.save(
    best_model.model.state_dict(),
    MODELS_DIR / "seq2seq_attention_mimo.pth",
)
print("salvo em", MODELS_DIR / "seq2seq_attention_mimo.pth")

salvo em ../../models/seq2seq_attention_mimo.pth
